# Ingest races.csv file
1. Read the file using dataframe reader API
2. Add Metadata Columns
    - Source File
    - Ingestion Timestamp
3. Write to bronze delta table 

In [0]:
from pyspark.sql.types import StructType, StructField,IntegerType,StringType,DoubleType

In [0]:
races_schema = StructType([
    StructField("raceId",IntegerType(),True),
    StructField("year",IntegerType(),True),
    StructField("round",IntegerType(),True),
    StructField("circuitId",IntegerType(),True),
    StructField("name",StringType(),True),
    StructField("date",StringType(),True),
    StructField("time",StringType(),True),
    StructField("url",StringType(),True)
])

In [0]:
races_df = (spark.read.format('csv')
            .option('header','true')
            .option('mode','FAILFAST')
            .schema(races_schema)
            .load('/Volumes/formula1/landing/files/races.csv')
            )

In [0]:
display(races_df)

In [0]:
from pyspark.sql.functions import current_timestamp,col

In [0]:
races_final_df = races_df.withColumn("ingestion_timestamp",current_timestamp())\
                         .withColumn("source_file",col("_metadata.file_path"))

In [0]:
display(races_final_df)

In [0]:
races_final_df.write.mode('overwrite').format('delta').saveAsTable('formula1.bronze.races')

In [0]:
display(spark.table('formula1.bronze.races'))